# Обучение CatBoost для ранжирования рекомендаций

Ноутбук загружает готовые Item2Vec-модели, вызывает генерацию кандидатов, обучает CatBoost и сравнивает популярный baseline, покрытие кандидатогенерации и итоговое ранжирование.

Пути к данным OTTO, готовым моделям Item2Vec и модулю генерации кандидатов заданы в `TRAIN_PATH`, `MODELS_DIR` и `CANDIDATE_MODULE_PATH`. В папке `/kaggle/working` сохраняются CatBoost-модель, список признаков, популярные товары и таблица сравнения.


In [1]:
!pip install faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.4 MB/s eta 0:00:00


In [2]:
import gc
import json
import random
import sys
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

import catboost as cb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import ujson

TRAIN_PATH = '/kaggle/input/competitions/otto-recommender-system/train.jsonl'
MODELS_DIR = Path(
    '/kaggle/input/notebooks/igorvorobev7892/recproj2-item2vec-training'
)

CANDIDATE_MODULE_PATH = Path(
    '/kaggle/input/notebooks/igorvorobev7892/recproj2-candidates/recproj2_candidates.py'
)

sys.path.insert(0, str(CANDIDATE_MODULE_PATH.parent))

from recproj2_candidates import (
    generate_candidates,
    load_model_and_build_index,
)

OUT_DIR = Path('/kaggle/working')
EVAL_SESSIONS_PATH = OUT_DIR / 'evaluation_sessions.jsonl'
TRAIN_SELECTION_PATH = OUT_DIR / 'train_selected_sessions.jsonl'
SAMPLE_SUMMARY_PATH = OUT_DIR / 'training_sample_summary.csv'
CANDIDATE_SUMMARY_PATH = OUT_DIR / 'training_candidates_summary.csv'

EVENT_TYPES = ['clicks', 'carts', 'orders']
EVENT_WEIGHTS = {'clicks': 0.10, 'carts': 0.30, 'orders': 0.60}
TYPE_WEIGHTS = {'clicks': 1.0, 'carts': 3.0, 'orders': 6.0}

DATA_END = datetime(2022, 8, 28, 21, 59, 59, 984000, tzinfo=timezone.utc)
VALIDATION_DAYS = 7
VALIDATION_START = DATA_END - timedelta(days=VALIDATION_DAYS)
VALIDATION_START_TS = int(VALIDATION_START.timestamp() * 1000)

MAX_TRAIN_SESSIONS = 1_200_000
MAX_TRAIN_EVAL_SESSIONS = 500_000
MAX_VALIDATION_EVAL_SESSIONS = 1_000_000
EARLY_STOP_SHARE = 0.10
SAMPLING_SEED = 11
TRAIN_GROUPS = ['orders', 'carts_without_orders', 'clicks_only']
TOP_K = 20
CANDIDATE_K = 150
TRAIN_CANDIDATE_K = 150
TOP_N_POPULAR = 200
LAST_N = 10
MAX_NEGATIVES_PER_POSITIVE = 3
MIN_NEGATIVES = 10
MAX_CANDIDATES_PER_SESSION = 100
MAX_BUFFER_ROWS = 50_000
EVAL_BATCH_SESSIONS = 250

random.seed(11)
np.random.seed(11)

print(f'Файл данных: {TRAIN_PATH}')
print(f'Папка Item2Vec-моделей: {MODELS_DIR}')
print(f'Начало validation: {VALIDATION_START.isoformat()}')

Файл данных: /kaggle/input/competitions/otto-recommender-system/train.jsonl
Папка Item2Vec-моделей: /kaggle/input/notebooks/igorvorobev7892/recproj2-item2vec-training
Начало validation: 2022-08-21T21:59:59.984000+00:00


## Временная граница и отбор сессий

Последние семь дней используются для validation. Популярность товаров и все обучающие примеры строятся только по событиям до этой границы.

В первом проходе каждая train-часть с минимум двумя событиями разделяется на историю и цели: первые 80% событий — история, оставшиеся — будущее. Группа определяется по целям:

| Группа | Действия в будущей части |
| --- | --- |
| `orders` | Есть хотя бы один заказ; другие действия могут присутствовать. |
| `carts_without_orders` | Есть добавление в корзину, заказов нет. |
| `clicks_only` | Только просмотры. |

Фильтра на обязательное наличие carts и orders во всей train-части больше нет. Например, заказ в истории и только click в будущем дают группу `clicks_only`.

Общий лимит — 1 200 000 сессий. Квоты сохраняют реальные доли трёх групп. Внутри каждой группы сохраняются доли сочетаний длины истории (1–5, 6–20 и от 21 события) и дня последнего события истории в UTC. Квоты округляются до целых с распределением остатка, поэтому при достаточном числе примеров выбирается ровно 1 200 000 сессий. При меньшем числе подходящих примеров используются все.

10% выбранных сессий выделяются для ранней остановки с теми же пропорциями. При полном лимите это 120 000 сессий для ранней остановки и 1 080 000 для обучения. Число для ранней остановки округляется вниз.

Validation-метрики используют до 1 000 000 сессий, в которых есть и carts, и orders начиная с границы (`ts >= VALIDATION_START_TS`). Оба действия могут относиться к разным товарам. Этот отбор выполняется отдельно от отбора для обучения.


In [3]:
def has_cart_and_order(events):
    event_types = {event['type'] for event in events}
    return 'carts' in event_types and 'orders' in event_types


def split_events(events):
    train_events = []
    validation_events = []
    for event in events:
        if int(event['ts']) < VALIDATION_START_TS:
            train_events.append(event)
        else:
            validation_events.append(event)
    return train_events, validation_events


def make_targets(future_events):
    targets = {event_type: set() for event_type in EVENT_TYPES}
    for event in future_events:
        event_type = event['type']
        aid = str(event['aid'])
        if event_type == 'clicks':
            if not targets['clicks']:
                targets['clicks'].add(aid)
        elif event_type in ('carts', 'orders'):
            targets[event_type].add(aid)
    return targets


def make_train_example(train_events):
    if len(train_events) < 2:
        return None

    split_index = max(1, len(train_events) * 4 // 5)
    history_events = train_events[:split_index]
    targets = make_targets(train_events[split_index:])
    if targets['orders']:
        target_group = 'orders'
    elif targets['carts']:
        target_group = 'carts_without_orders'
    else:
        target_group = 'clicks_only'

    history_len = len(history_events)
    if history_len <= 5:
        length_group = '1-5'
    elif history_len <= 20:
        length_group = '6-20'
    else:
        length_group = '21+'
    prediction_day = int(history_events[-1]['ts']) // 86_400_000
    stratum = (target_group, length_group, prediction_day)
    return history_events, targets, stratum


def proportional_quotas(counts, sample_size):
    """Целые квоты с сохранением долей и точным суммарным размером."""
    total = sum(counts.values())
    sample_size = min(sample_size, total)
    quotas = {key: 0 for key in counts}
    if total == 0:
        return quotas

    remainders = {}
    for key in counts:
        quotas[key], remainders[key] = divmod(counts[key] * sample_size, total)
    extra = sample_size - sum(quotas.values())
    for key in sorted(counts, key=lambda key: (-remainders[key], key))[:extra]:
        quotas[key] += 1
    return quotas


def stratified_quotas(counts, sample_size):
    """Сначала доли будущих действий, затем длины истории и дня внутри группы."""
    group_counts = {group: 0 for group in TRAIN_GROUPS}
    for (group, length_group, day), count in counts.items():
        group_counts[group] += count
    group_quotas = proportional_quotas(group_counts, sample_size)
    quotas = {}
    for group in TRAIN_GROUPS:
        group_strata = {key: count for key, count in counts.items() if key[0] == group}
        quotas.update(proportional_quotas(group_strata, group_quotas[group]))
    return quotas


def take_from_pool(key, remaining_pool, remaining_quota, rng):
    """Случайный отбор без повторов: нужно выбрать / осталось просмотреть."""
    selected = rng.randrange(remaining_pool[key]) < remaining_quota[key]
    remaining_pool[key] -= 1
    remaining_quota[key] -= int(selected)
    return selected


popularity_counts = {event_type: Counter() for event_type in EVENT_TYPES}
train_stratum_counts = Counter()
train_event_count = 0
validation_event_count = 0
train_sessions_count = 0
validation_sessions_count = 0
validation_sessions_with_cart_and_order = 0
sessions_crossing_boundary = 0
total_sessions = 0

with open(TRAIN_PATH, 'r') as train_file:
    for session_index, line in enumerate(train_file):
        total_sessions += 1
        if session_index and session_index % 1_000_000 == 0:
            print(f'Прочитано сессий: {session_index:,}')

        events = ujson.loads(line)['events']
        train_events, validation_events = split_events(events)
        train_event_count += len(train_events)
        validation_event_count += len(validation_events)
        train_sessions_count += int(bool(train_events))
        validation_sessions_count += int(bool(validation_events))
        validation_sessions_with_cart_and_order += int(
            has_cart_and_order(validation_events)
        )
        sessions_crossing_boundary += int(bool(train_events and validation_events))

        for event in train_events:
            popularity_counts[event['type']][str(event['aid'])] += 1

        example = make_train_example(train_events)
        if example is not None:
            train_stratum_counts[example[2]] += 1

eligible_train_sessions_count = sum(train_stratum_counts.values())
selected_train_count = min(eligible_train_sessions_count, MAX_TRAIN_SESSIONS)
train_selection_quotas = stratified_quotas(train_stratum_counts, selected_train_count)
early_stop_count = int(selected_train_count * EARLY_STOP_SHARE)
early_stop_quotas = stratified_quotas(train_selection_quotas, early_stop_count)
fit_quotas = {
    key: train_selection_quotas[key] - early_stop_quotas[key]
    for key in train_selection_quotas
}
fit_session_count = selected_train_count - early_stop_count
train_eval_sample_size = min(fit_session_count, MAX_TRAIN_EVAL_SESSIONS)
train_eval_quotas = stratified_quotas(fit_quotas, train_eval_sample_size)
validation_eval_sample_size = min(
    validation_sessions_with_cart_and_order, MAX_VALIDATION_EVAL_SESSIONS
)

popular_items = {
    event_type: [aid for aid, _ in counts.most_common(TOP_N_POPULAR)]
    for event_type, counts in popularity_counts.items()
}
del popularity_counts, events, train_events, validation_events, example
gc.collect()

with open(OUT_DIR / 'popular_items.json', 'w') as output_file:
    json.dump(popular_items, output_file)

planned_sampling = pd.DataFrame([
    {
        'target_group': group,
        'available_sessions': sum(n for key, n in train_stratum_counts.items() if key[0] == group),
        'selected_sessions': sum(n for key, n in train_selection_quotas.items() if key[0] == group),
        'fit_sessions': sum(n for key, n in fit_quotas.items() if key[0] == group),
        'early_stop_sessions': sum(n for key, n in early_stop_quotas.items() if key[0] == group),
    }
    for group in TRAIN_GROUPS
])

print(f'Всего сессий: {total_sessions:,}')
print(f'Train: {train_sessions_count:,} сессий, {train_event_count:,} событий')
print(f'Подходят для разреза на историю и цели: {eligible_train_sessions_count:,}')
print(
    f'Выборка CatBoost: {selected_train_count:,}; '
    f'обучение — {fit_session_count:,}; ранняя остановка — {early_stop_count:,}.'
)
print(
    f'Validation: {validation_sessions_count:,} сессий, '
    f'{validation_event_count:,} событий; с carts и orders после границы — '
    f'{validation_sessions_with_cart_and_order:,}.'
)
print(
    f'Для метрик: train — {train_eval_sample_size:,}; '
    f'validation — {validation_eval_sample_size:,}.'
)
print(f'Сессий по обе стороны границы: {sessions_crossing_boundary:,}')
display(planned_sampling)


Прочитано сессий: 1,000,000
Прочитано сессий: 2,000,000
Прочитано сессий: 3,000,000
Прочитано сессий: 4,000,000
Прочитано сессий: 5,000,000
Прочитано сессий: 6,000,000
Прочитано сессий: 7,000,000
Прочитано сессий: 8,000,000
Прочитано сессий: 9,000,000
Прочитано сессий: 10,000,000
Прочитано сессий: 11,000,000
Прочитано сессий: 12,000,000
Всего сессий: 12,899,779
Train: 11,098,528 сессий, 163,955,180 событий
Подходят для разреза на историю и цели: 10,584,517
Выборка CatBoost: 1,200,000; обучение — 1,080,000; ранняя остановка — 120,000.
Validation: 5,323,084 сессий, 52,760,916 событий; с carts и orders после границы — 471,510.
Для метрик: train — 500,000; validation — 471,510.
Сессий по обе стороны границы: 3,521,833


,target_group,available_sessions,selected_sessions,fit_sessions,early_stop_sessions
0,orders,858070,97282,87554,9728
1,carts_without_orders,983410,111492,100343,11149
2,clicks_only,8743037,991226,892103,99123


## Кандидаты и признаки

Функции `load_model_and_build_index` и `generate_candidates` импортированы из `recproj2_candidates.py`. Модель Item2Vec и FAISS-индекс создаются заранее и используются для всех сессий. Вызов `generate_candidates(..., return_metadata=True)` возвращает список товаров и их оценки с источниками.

Ниже рассчитывается одна строка признаков для каждой пары «известная история сессии — товар-кандидат». Все действия, доли и интервалы берутся из переданной истории.

### 1. Действия с кандидатом

Эта группа описывает повторный интерес к товару и последовательность действий с ним. Отдельные счётчики позволяют различать повторные просмотры, добавления в корзину и заказы.

| Признаки | Что рассчитывается и зачем |
| --- | --- |
| `item_click_count`, `item_cart_count`, `item_order_count` | Число действий каждого типа с кандидатом. Показывает интенсивность интереса и покупок. |
| `item_last_action` | Последнее известное действие с кандидатом. Различает истории, заканчивающиеся просмотром, добавлением в корзину или заказом. |
| `item_click_share` | Доля просмотров кандидата среди всех просмотров в истории. Показывает, насколько внимание сосредоточено на одном товаре. |
| `item_click_count_last_5` | Число просмотров кандидата среди последних пяти событий любого типа. Выделяет недавний интерес внутри длинной истории. |
| `item_cart_after_last_order` | Равен 1, если последний cart кандидата находится после последнего order. При отсутствии order достаточно наличия cart. Показывает добавление в корзину, после которого в известной истории ещё не было заказа этого товара. |

### 2. Время и недавняя активность

Позиция в последовательности дополняется реальным временем: соседние события могут быть разделены длительной паузой. Эти признаки описывают давность интереса и темп действий.

| Признаки | Что рассчитывается и зачем |
| --- | --- |
| `seconds_since_item_last_event` | Секунды после последнего взаимодействия с кандидатом до момента прогноза. |
| `seconds_since_item_last_click`, `seconds_since_item_last_cart` | Давность просмотра и добавления кандидата в корзину по отдельности. |
| `history_duration_seconds` | Секунды между первым и последним известным событием. Описывает длительность наблюдаемой истории. |
| `last_event_gap_seconds` | Пауза между двумя последними известными событиями. Описывает темп последних действий. |
| `history_events_last_hour`, `history_events_last_day` | Число известных событий за последний час и последние сутки перед моментом прогноза. Выделяет текущую активность. |

Для train момент прогноза совпадает с последним событием доступной истории. Для validation это `VALIDATION_START_TS`. Границы часового и суточного интервалов включаются в подсчёт. Если соответствующего действия нет, давность равна `-1`. Пауза между последними событиями равна `-1`, если их меньше двух.

### 3. Состояние сессии

Эта группа описывает общую активность и сочетание действий в истории. Признаки одинаковы для всех кандидатов одной сессии и позволяют модели учитывать признаки товара вместе с характером сессии.

| Признаки | Что рассчитывается и зачем |
| --- | --- |
| `history_click_count`, `history_cart_count`, `history_order_count` | Общие числа просмотров, добавлений и заказов в известной истории. |
| `history_cart_share`, `history_order_share` | Доли cart и order среди всех известных событий. Позволяют сопоставлять истории разной длины. |
| `history_last_action` | Последнее известное действие в сессии. Описывает её текущее состояние. |
| `history_unique_cart_items` | Число разных товаров, которые добавляли в корзину. Описывает широту выбора. |
| `has_history` | Наличие известных событий. Выделяет примеры, для которых доступны только кандидаты из популярного списка. |

Для пустой истории счётчики, доли, длительность и `repeat_ratio` равны `0`, последние действия обозначаются строкой `none`. `item_last_action`, `history_last_action` и целевой `event_type` передаются CatBoost как категориальные признаки.

### 4. Оценки и источники кандидатов

Генератор передаёт числовые оценки и происхождение товара. Это позволяет CatBoost учитывать силу каждого сигнала вместе с историей сессии.

| Признаки | Что рассчитывается и зачем |
| --- | --- |
| `history_score` | Сумма весов действий с кандидатом с учётом их относительного времени в истории. Описывает оценку товара до добавления соседей Item2Vec. |
| `item2vec_retrieval_score` | Исходное косинусное сходство кандидата с вектором последних событий при поиске соседей. Показывает близость к недавнему интересу. |
| `from_history`, `from_item2vec`, `from_popular` | Флаги источников, предложивших товар при данном вызове генератора. Источники могут пересекаться. Популярный список используется для дополнения кандидатов и при пустой истории. |
| `candidate_source_count` | Сумма флагов источников. Показывает, сколько источников предложило товар. |

При отсутствии источника его оценка равна `0`; флаг позволяет отличить отсутствие от нулевой оценки. Перед отбором отрицательных примеров сохраняются оценки и исходные позиции полного списка кандидатов.

### 5. Положение, популярность и близость Item2Vec

`candidate_rank` и `candidate_rank_norm` описывают позицию в исходном списке. `first_pos`, `last_pos`, `pos_normalized` и `distance_from_end` описывают положение товара в истории. `in_session`, флаги действий, `session_len`, `session_unique` и `repeat_ratio` дополняют информацию о повторениях.

`pop_rank`, `pop_score`, `in_top_10` и `in_top_50` описывают позицию в популярном списке соответствующего типа событий. `i2v_session_sim` и `i2v_last_sim` показывают близость кандидата ко всей известной истории и последнему товару.

Счётчики и служебные оценки создаются для текущей сессии. Готовые строки записываются порциями; числовые признаки хранятся в `float32` и `int32`.


In [4]:
def compute_features_for_session(session_aids, session_types, session_ts,
                                 candidates, event_type, model, *,
                                 prediction_ts, candidate_metadata,
                                 candidate_rank_map=None):
    """Признаки пары «сессия — товар» по известной истории; время задано в мс."""
    session_len = len(session_aids)
    unique_items = len(set(session_aids))
    history_counts = Counter()
    first_pos = {}
    last_pos = {}
    item_action_counts = defaultdict(Counter)
    item_last_action = {}
    item_last_event_ts = {}
    item_last_click_ts = {}
    item_last_cart_ts = {}
    item_last_cart_pos = {}
    item_last_order_pos = {}
    events_last_hour = 0
    events_last_day = 0

    # За один проход считаются действия, позиции и давность событий.
    hour_start = prediction_ts - 60 * 60 * 1000
    day_start = prediction_ts - 24 * 60 * 60 * 1000
    for position, (aid, action, ts) in enumerate(
        zip(session_aids, session_types, session_ts)
    ):
        history_counts[action] += 1
        item_action_counts[aid][action] += 1
        first_pos.setdefault(aid, position)
        last_pos[aid] = position
        item_last_action[aid] = action
        item_last_event_ts[aid] = ts
        events_last_hour += int(hour_start <= ts <= prediction_ts)
        events_last_day += int(day_start <= ts <= prediction_ts)

        if action == 'clicks':
            item_last_click_ts[aid] = ts
        elif action == 'carts':
            item_last_cart_ts[aid] = ts
            item_last_cart_pos[aid] = position
        elif action == 'orders':
            item_last_order_pos[aid] = position

    clicks_last_5 = Counter(
        aid for aid, action in zip(session_aids[-5:], session_types[-5:])
        if action == 'clicks'
    )

    session_features = {
        'session_len': session_len,
        'session_unique': unique_items,
        'repeat_ratio': 1.0 - unique_items / session_len if session_len else 0.0,
        'has_history': int(session_len > 0),
        'history_click_count': history_counts['clicks'],
        'history_cart_count': history_counts['carts'],
        'history_order_count': history_counts['orders'],
        'history_cart_share': history_counts['carts'] / max(session_len, 1),
        'history_order_share': history_counts['orders'] / max(session_len, 1),
        'history_last_action': session_types[-1] if session_len else 'none',
        'history_unique_cart_items': len(item_last_cart_pos),
        'history_duration_seconds': (
            (session_ts[-1] - session_ts[0]) / 1000.0 if session_len else 0.0
        ),
        'last_event_gap_seconds': (
            (session_ts[-1] - session_ts[-2]) / 1000.0
            if session_len >= 2 else -1.0
        ),
        'history_events_last_hour': events_last_hour,
        'history_events_last_day': events_last_day,
    }

    session_vector = None
    if session_aids:
        min_ts = session_ts[0]
        max_ts = session_ts[-1]
        time_span = max(max_ts - min_ts, 1.0)
        history_vectors = []
        history_weights = []

        for aid, action, ts in zip(session_aids, session_types, session_ts):
            if aid in model.wv.key_to_index:
                time_weight = (ts - min_ts) / time_span
                history_vectors.append(model.wv[aid])
                history_weights.append(
                    TYPE_WEIGHTS.get(action, 1.0) * time_weight
                )

        if history_vectors:
            history_weights = np.asarray(history_weights, dtype=np.float32)
            weight_sum = history_weights.sum()
            if weight_sum > 0:
                history_weights /= weight_sum
            else:
                history_weights[:] = 1.0 / len(history_weights)
            session_vector = np.average(
                history_vectors, axis=0, weights=history_weights
            )
            norm = np.linalg.norm(session_vector)
            if norm > 0:
                session_vector /= norm

    valid_candidates = [
        aid for aid in candidates if aid in model.wv.key_to_index
    ]
    candidate_vectors = [model.wv[aid] for aid in valid_candidates]
    session_similarities = {}
    last_item_similarities = {}

    if candidate_vectors:
        candidate_matrix = np.asarray(candidate_vectors, dtype=np.float32)
        norms = np.linalg.norm(candidate_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        candidate_matrix /= norms

        if session_vector is not None:
            similarities = candidate_matrix @ session_vector
            session_similarities = dict(zip(
                valid_candidates, similarities.astype(float)
            ))

        if session_aids:
            last_aid = session_aids[-1]
            if last_aid in model.wv.key_to_index:
                last_vector = model.wv[last_aid]
                last_norm = np.linalg.norm(last_vector)
                if last_norm > 0:
                    similarities = candidate_matrix @ (last_vector / last_norm)
                    last_item_similarities = dict(zip(
                        valid_candidates, similarities.astype(float)
                    ))

    pop_list = popular_items.get(event_type, [])
    pop_rank = {aid: rank for rank, aid in enumerate(pop_list)}
    candidate_rank = candidate_rank_map if candidate_rank_map is not None else {
        aid: rank for rank, aid in enumerate(candidates)
    }

    def seconds_since(last_ts):
        return (prediction_ts - last_ts) / 1000.0 if last_ts is not None else -1.0

    rows = []
    for aid in candidates:
        action_counts = item_action_counts.get(aid, {})
        click_count = action_counts.get('clicks', 0)
        cart_count = action_counts.get('carts', 0)
        order_count = action_counts.get('orders', 0)
        rank = candidate_rank[aid]
        popularity_rank = pop_rank.get(aid, 999)
        row = {
            'candidate_rank': rank,
            'candidate_rank_norm': rank / max(CANDIDATE_K - 1, 1),
            'in_session': int(aid in last_pos),
            'count_in_session': click_count + cart_count + order_count,
            'first_pos': first_pos.get(aid, -1),
            'last_pos': last_pos.get(aid, -1),
            'was_clicked': int(click_count > 0),
            'was_carted': int(cart_count > 0),
            'was_ordered': int(order_count > 0),
            'type_count': len(action_counts),
            'item_click_count': click_count,
            'item_cart_count': cart_count,
            'item_order_count': order_count,
            'item_last_action': item_last_action.get(aid, 'none'),
            'item_click_share': click_count / max(history_counts['clicks'], 1),
            'item_click_count_last_5': clicks_last_5.get(aid, 0),
            'item_cart_after_last_order': int(
                item_last_cart_pos.get(aid, -1) > item_last_order_pos.get(aid, -1)
            ),
            'seconds_since_item_last_event': seconds_since(item_last_event_ts.get(aid)),
            'seconds_since_item_last_click': seconds_since(item_last_click_ts.get(aid)),
            'seconds_since_item_last_cart': seconds_since(item_last_cart_ts.get(aid)),
            'pop_rank': popularity_rank,
            'pop_score': (
                1.0 - popularity_rank / max(len(pop_list), 1)
                if aid in pop_rank else 0.0
            ),
            'in_top_10': int(popularity_rank < 10),
            'in_top_50': int(popularity_rank < 50),
            'pos_normalized': (
                last_pos[aid] / max(session_len - 1, 1)
                if aid in last_pos else -1.0
            ),
            'is_last_item': int(aid in last_pos and last_pos[aid] == session_len - 1),
            'is_first_item': int(aid in first_pos and first_pos[aid] == 0),
            'distance_from_end': session_len - 1 - last_pos[aid] if aid in last_pos else -1,
            'i2v_session_sim': session_similarities.get(aid, 0.0),
            'i2v_last_sim': last_item_similarities.get(aid, 0.0),
            'event_type': event_type,
        }
        row.update(session_features)
        row.update(candidate_metadata[aid])
        rows.append(row)
    return rows


## Подготовка обучающих примеров

Во втором проходе сессии случайно отбираются внутри рассчитанных групп. Вероятность выбора равна числу оставшихся мест, делённому на число ещё не просмотренных сессий этой группы. Это позволяет заполнить квоты без повторов и без хранения всего датасета в памяти. С фиксированным `SAMPLING_SEED = 11` и теми же данными выборка воспроизводится.

Для отбора сессий, ранней остановки, train-метрик и негативов используются отдельные генераторы случайных чисел. Изменение числа негативов не меняет выбранные сессии. Сначала определяется часть сессии (`train` или `early_stop`), затем создаются все её строки признаков: кандидаты и типы действий одной сессии не смешиваются между этими частями.

Для clicks цель — первый click в будущем, для carts и orders — все уникальные будущие товары. Группа будущих действий используется только для отбора и отчёта, в признаки CatBoost она не передаётся.

Кандидаты генерируются только по известной истории. Как и раньше, группа «сессия — тип действия» без позитивов среди кандидатов не даёт обучающих строк. В отчёте отдельно показаны число сессий с целью, число с найденным позитивом и количество положительных и отрицательных строк. Лимит сессий относится к выборке до этого этапа, поэтому фактическое число сессий со строками признаков может быть меньше.

`train_selected_sessions.jsonl` сохраняет ID выбранных сессий, их группу, часть и участие в train-метриках. `training_sample_summary.csv` показывает распределение по группам, длине и дням, а `training_candidates_summary.csv` — фактические результаты кандидатогенерации.

Для train-метрик отдельно выбирается до 500 000 сессий только из части `train`, с сохранением её пропорций. Сессии для ранней остановки туда не входят. Истории и цели train и validation записываются в `evaluation_sessions.jsonl`, чтобы все этапы оценивались на одинаковых примерах.


In [5]:
def downcast_frame(frame):
    for column in frame.select_dtypes(include=['float', 'float64']).columns:
        frame[column] = frame[column].astype(np.float32)
    for column in frame.select_dtypes(include=['integer']).columns:
        dtype = np.int8 if column == 'label' else np.int32
        frame[column] = frame[column].astype(dtype)
    return frame


def selected_for_evaluation(session_number, total_sessions, sample_size):
    """Равномерный отбор заданного числа сессий; нумерация начинается с 1."""
    return (
        session_number * sample_size // total_sessions
        > (session_number - 1) * sample_size // total_sessions
    )


def save_evaluation_session(output_file, split_name, session_id,
                            history_events, targets_by_type, prediction_ts):
    record = {
        'split': split_name,
        'session': session_id,
        'history': history_events,
        'targets': {
            event_type: sorted(targets)
            for event_type, targets in targets_by_type.items()
        },
        'prediction_ts': prediction_ts,
    }
    output_file.write(ujson.dumps(record) + '\n')
    evaluation_session_counts[split_name] += 1


TRAIN_FEATURES_PATH = OUT_DIR / 'train_features.parquet'
EARLY_STOP_FEATURES_PATH = OUT_DIR / 'early_stop_features.parquet'

parquet_paths = {
    'train': TRAIN_FEATURES_PATH,
    'early_stop': EARLY_STOP_FEATURES_PATH,
}
parquet_writers = {'train': None, 'early_stop': None}
row_buffers = {'train': [], 'early_stop': []}
train_batch_rows = {'train': 0, 'early_stop': 0}


def flush_feature_rows(split_name):
    rows = row_buffers[split_name]
    if not rows:
        return

    frame = downcast_frame(pd.DataFrame.from_records(rows))
    table = pa.Table.from_pandas(frame, preserve_index=False)
    writer = parquet_writers[split_name]
    if writer is None:
        writer = pq.ParquetWriter(
            str(parquet_paths[split_name]),
            table.schema,
            compression='snappy',
        )
        parquet_writers[split_name] = writer
    writer.write_table(table)
    train_batch_rows[split_name] += len(frame)
    row_buffers[split_name] = []
    del frame, table
    gc.collect()


item2vec_models = {}
indexes = {}
idx_to_aids = {}
for event_type in EVENT_TYPES:
    model, index, idx_to_aid = load_model_and_build_index(
        MODELS_DIR / f'item2vec_{event_type}.model'
    )
    item2vec_models[event_type] = model
    indexes[event_type] = index
    idx_to_aids[event_type] = idx_to_aid
    del model, index, idx_to_aid
    gc.collect()
    print(f'Загружена Item2Vec-модель: {event_type}')

selection_rng = random.Random(SAMPLING_SEED)
early_stop_rng = random.Random(SAMPLING_SEED + 1)
train_eval_rng = random.Random(SAMPLING_SEED + 2)
negative_rng = random.Random(SAMPLING_SEED + 3)
remaining_train_pool = dict(train_stratum_counts)
remaining_train_quota = dict(train_selection_quotas)
remaining_early_pool = dict(train_selection_quotas)
remaining_early_quota = dict(early_stop_quotas)
remaining_eval_pool = dict(fit_quotas)
remaining_eval_quota = dict(train_eval_quotas)

selected_train_sessions = 0
validation_session_offset = 0
selection_counts = {'train': Counter(), 'early_stop': Counter()}
useful_session_counts = {'train': Counter(), 'early_stop': Counter()}
generation_counts = {
    (split_name, group, event_type): Counter()
    for split_name in ('train', 'early_stop')
    for group in TRAIN_GROUPS
    for event_type in EVENT_TYPES
}
evaluation_session_counts = {'train_sample': 0, 'validation': 0}

with (
    open(TRAIN_PATH, 'r') as train_file,
    open(EVAL_SESSIONS_PATH, 'w') as eval_file,
    open(TRAIN_SELECTION_PATH, 'w') as selection_file,
):
    for session_index, line in enumerate(train_file):
        if session_index and session_index % 1_000_000 == 0:
            print(
                f'Прочитано {session_index:,} сессий; '
                f'выбрано для CatBoost {selected_train_sessions:,}.'
            )
        session = ujson.loads(line)
        train_events, validation_events = split_events(session['events'])

        if has_cart_and_order(validation_events):
            validation_session_offset += 1
            if selected_for_evaluation(
                validation_session_offset,
                validation_sessions_with_cart_and_order,
                validation_eval_sample_size,
            ):
                save_evaluation_session(
                    eval_file, 'validation', session['session'],
                    train_events, make_targets(validation_events),
                    prediction_ts=VALIDATION_START_TS,
                )

        example = make_train_example(train_events)
        if example is None:
            continue
        history_events, targets_by_type, stratum = example
        if not take_from_pool(
            stratum, remaining_train_pool, remaining_train_quota, selection_rng
        ):
            continue

        selected_train_sessions += 1
        target_group, length_group, prediction_day = stratum
        is_early_stop = take_from_pool(
            stratum, remaining_early_pool, remaining_early_quota, early_stop_rng
        )
        split_name = 'early_stop' if is_early_stop else 'train'
        selection_counts[split_name][stratum] += 1

        for_train_eval = False
        if split_name == 'train':
            for_train_eval = take_from_pool(
                stratum, remaining_eval_pool, remaining_eval_quota, train_eval_rng
            )
            if for_train_eval:
                save_evaluation_session(
                    eval_file, 'train_sample', session['session'],
                    history_events, targets_by_type,
                    prediction_ts=float(history_events[-1]['ts']),
                )

        selection_file.write(ujson.dumps({
            'session': session['session'],
            'split': split_name,
            'target_group': target_group,
            'history_length': len(history_events),
            'length_group': length_group,
            'prediction_ts': int(history_events[-1]['ts']),
            'for_train_eval': for_train_eval,
        }) + '\n')

        history_aids = [str(event['aid']) for event in history_events]
        history_types = [event['type'] for event in history_events]
        history_ts = [float(event['ts']) for event in history_events]
        session_has_training_rows = False

        for event_type in EVENT_TYPES:
            positive_set = targets_by_type[event_type]
            if not positive_set:
                continue

            counts = generation_counts[(split_name, target_group, event_type)]
            counts['sessions_with_target'] += 1
            counts['target_items'] += len(positive_set)
            candidates, candidate_metadata = generate_candidates(
                history_aids, history_types, history_ts, event_type,
                item2vec_models[event_type], indexes[event_type],
                idx_to_aids[event_type],
                popular_items=popular_items, type_weights=TYPE_WEIGHTS,
                top_k=TRAIN_CANDIDATE_K, last_n=LAST_N,
                return_metadata=True,
            )
            candidate_rank_map = {
                aid: rank for rank, aid in enumerate(candidates)
            }
            positive_aids = [aid for aid in candidates if aid in positive_set]
            if not positive_aids:
                continue
            counts['sessions_with_positive_candidate'] += 1
            counts['positive_rows'] += len(positive_aids)

            negative_aids = [aid for aid in candidates if aid not in positive_set]
            negative_count = max(
                len(positive_aids) * MAX_NEGATIVES_PER_POSITIVE,
                MIN_NEGATIVES,
            )
            if len(negative_aids) > negative_count:
                negative_aids = negative_rng.sample(negative_aids, negative_count)

            negative_capacity = max(
                0, MAX_CANDIDATES_PER_SESSION - len(positive_aids)
            )
            if len(negative_aids) > negative_capacity:
                negative_aids = negative_rng.sample(negative_aids, negative_capacity)
            counts['negative_rows'] += len(negative_aids)

            retained = set(positive_aids + negative_aids)
            balanced_candidates = [
                aid for aid in candidates if aid in retained
            ]
            feature_rows = compute_features_for_session(
                history_aids, history_types, history_ts,
                balanced_candidates, event_type, item2vec_models[event_type],
                prediction_ts=history_ts[-1],
                candidate_metadata=candidate_metadata,
                candidate_rank_map=candidate_rank_map,
            )
            for row, aid in zip(feature_rows, balanced_candidates):
                row['label'] = int(aid in positive_set)
            row_buffers[split_name].extend(feature_rows)
            session_has_training_rows = True

            if len(row_buffers[split_name]) >= MAX_BUFFER_ROWS:
                flush_feature_rows(split_name)

        useful_session_counts[split_name][stratum] += int(session_has_training_rows)

for split_name in row_buffers:
    flush_feature_rows(split_name)
for writer in parquet_writers.values():
    if writer is not None:
        writer.close()

sample_summary_rows = []
for stratum, available in sorted(train_stratum_counts.items()):
    group, length_group, day = stratum
    sample_summary_rows.append({
        'target_group': group,
        'length_group': length_group,
        'prediction_date_utc': datetime.fromtimestamp(day * 86_400, timezone.utc).date().isoformat(),
        'available_sessions': available,
        'planned_selected_sessions': train_selection_quotas[stratum],
        'fit_sessions': selection_counts['train'][stratum],
        'early_stop_sessions': selection_counts['early_stop'][stratum],
        'fit_sessions_with_rows': useful_session_counts['train'][stratum],
        'early_stop_sessions_with_rows': useful_session_counts['early_stop'][stratum],
    })
sample_summary = pd.DataFrame(sample_summary_rows)
sample_summary.to_csv(SAMPLE_SUMMARY_PATH, index=False)

candidate_summary_rows = []
for (split_name, group, event_type), counts in generation_counts.items():
    candidate_summary_rows.append({
        'split': split_name,
        'target_group': group,
        'event_type': event_type,
        'selected_sessions': sum(n for key, n in selection_counts[split_name].items() if key[0] == group),
        'sessions_with_target': counts['sessions_with_target'],
        'sessions_with_positive_candidate': counts['sessions_with_positive_candidate'],
        'sessions_without_positive_candidate': (
            counts['sessions_with_target'] - counts['sessions_with_positive_candidate']
        ),
        'target_items': counts['target_items'],
        'positive_rows': counts['positive_rows'],
        'negative_rows': counts['negative_rows'],
    })
candidate_summary = pd.DataFrame(candidate_summary_rows)
candidate_summary.to_csv(CANDIDATE_SUMMARY_PATH, index=False)

print(f'Выбрано сессий: {selected_train_sessions:,}')
for split_name in ('train', 'early_stop'):
    print(
        f'{split_name}: выбрано {sum(selection_counts[split_name].values()):,}; '
        f'дали обучающие строки {sum(useful_session_counts[split_name].values()):,}; '
        f'строк признаков {train_batch_rows[split_name]:,}.'
    )
display(candidate_summary)
print(
    f'Сессий для итоговых метрик: '
    f'train — {evaluation_session_counts["train_sample"]:,}, '
    f'validation — {evaluation_session_counts["validation"]:,}.'
)
print(f'Выборки для метрик сохранены: {EVAL_SESSIONS_PATH}')
print(f'Список отобранных сессий: {TRAIN_SELECTION_PATH}')
print(f'Распределение выборки: {SAMPLE_SUMMARY_PATH}')
print(f'Статистика кандидатогенерации: {CANDIDATE_SUMMARY_PATH}')
del item2vec_models, indexes, idx_to_aids
gc.collect()


Загружена Item2Vec-модель: clicks
Загружена Item2Vec-модель: carts
Загружена Item2Vec-модель: orders
Прочитано 1,000,000 сессий; выбрано для CatBoost 110,867.
Прочитано 2,000,000 сессий; выбрано для CatBoost 221,882.
Прочитано 3,000,000 сессий; выбрано для CatBoost 332,057.
Прочитано 4,000,000 сессий; выбрано для CatBoost 442,016.
Прочитано 5,000,000 сессий; выбрано для CatBoost 551,223.
Прочитано 6,000,000 сессий; выбрано для CatBoost 660,613.
Прочитано 7,000,000 сессий; выбрано для CatBoost 768,913.
Прочитано 8,000,000 сессий; выбрано для CatBoost 876,920.
Прочитано 9,000,000 сессий; выбрано для CatBoost 984,324.
Прочитано 10,000,000 сессий; выбрано для CatBoost 1,089,619.
Прочитано 11,000,000 сессий; выбрано для CatBoost 1,190,727.
Прочитано 12,000,000 сессий; выбрано для CatBoost 1,200,000.
Выбрано сессий: 1,200,000
train: выбрано 1,080,000; дали обучающие строки 594,527; строк признаков 7,645,843.
early_stop: выбрано 120,000; дали обучающие строки 66,192; строк признаков 851,647.


,split,target_group,event_type,selected_sessions,sessions_with_target,sessions_with_positive_candidate,sessions_without_positive_candidate,target_items,positive_rows,negative_rows
0,train,orders,clicks,87554,65915,36082,29833,65915,36082,360820
1,train,orders,carts,87554,49587,31276,18311,85368,38768,315228
2,train,orders,orders,87554,87554,71129,16425,183398,128642,762511
3,train,carts_without_orders,clicks,100343,85926,42758,43168,85926,42758,427580
4,train,carts_without_orders,carts,100343,100343,58605,41738,147599,65827,587917
5,train,carts_without_orders,orders,100343,0,0,0,0,0,0
6,train,clicks_only,clicks,892103,892103,443610,448493,892103,443610,4436100
7,train,clicks_only,carts,892103,0,0,0,0,0,0
8,train,clicks_only,orders,892103,0,0,0,0,0,0
9,early_stop,orders,clicks,9728,7351,3974,3377,7351,3974,39740


Сессий для итоговых метрик: train — 500,000, validation — 471,510.
Выборки для метрик сохранены: /kaggle/working/evaluation_sessions.jsonl
Список отобранных сессий: /kaggle/working/train_selected_sessions.jsonl
Распределение выборки: /kaggle/working/training_sample_summary.csv
Статистика кандидатогенерации: /kaggle/working/training_candidates_summary.csv


0

## Обучение CatBoost

CatBoost обучается на кандидатах из сессий части `train`. Для ранней остановки используется отдельная случайная выборка целых сессий `early_stop` с сохранением пропорций групп; последняя неделя не участвует в обучении.


In [6]:
train_frame = downcast_frame(pd.read_parquet(TRAIN_FEATURES_PATH))
early_stop_frame = downcast_frame(pd.read_parquet(EARLY_STOP_FEATURES_PATH))

feature_cols = [column for column in train_frame.columns if column != 'label']
cat_features = ['event_type', 'item_last_action', 'history_last_action']

train_weights = train_frame['event_type'].map(TYPE_WEIGHTS).to_numpy(
    dtype=np.float32
)
train_pool = cb.Pool(
    train_frame[feature_cols],
    train_frame['label'],
    cat_features=cat_features,
    weight=train_weights,
)
del train_frame, train_weights
gc.collect()

early_stop_weights = early_stop_frame['event_type'].map(TYPE_WEIGHTS).to_numpy(
    dtype=np.float32
)
early_stop_pool = cb.Pool(
    early_stop_frame[feature_cols],
    early_stop_frame['label'],
    cat_features=cat_features,
    weight=early_stop_weights,
)
del early_stop_frame, early_stop_weights
gc.collect()

print(f'Строк для обучения: {train_pool.num_row():,}')
print(f'Строк для ранней остановки: {early_stop_pool.num_row():,}')

catboost_model = cb.CatBoostClassifier(
    iterations=1_000,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=11,
    cat_features=cat_features,
    verbose=100,
    thread_count=4,
    l2_leaf_reg=3.0,
    bootstrap_type='Bernoulli',
    subsample=0.8,
    colsample_bylevel=0.8,
    border_count=128,
    allow_writing_files=False,
)
catboost_model.fit(
    train_pool,
    eval_set=early_stop_pool,
    early_stopping_rounds=50,
    use_best_model=True,
    plot=False,
)

MODEL_PATH = OUT_DIR / 'catboost_ranker.cbm'
FEATURES_PATH = OUT_DIR / 'feature_cols.json'
CATEGORICAL_FEATURES_PATH = OUT_DIR / 'categorical_features.json'
IMPORTANCE_PATH = OUT_DIR / 'catboost_feature_importance.csv'

catboost_model.save_model(str(MODEL_PATH))
with open(FEATURES_PATH, 'w') as feature_file:
    json.dump(feature_cols, feature_file, ensure_ascii=False, indent=2)
with open(CATEGORICAL_FEATURES_PATH, 'w') as feature_file:
    json.dump(cat_features, feature_file, ensure_ascii=False, indent=2)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': catboost_model.feature_importances_,
}).sort_values('importance', ascending=False)
feature_importance.to_csv(IMPORTANCE_PATH, index=False)

print('Лучшая итерация:', catboost_model.get_best_iteration())
print('Метрики обучения:', catboost_model.get_best_score())
print(f'Модель сохранена: {MODEL_PATH}')
print(f'Список признаков сохранён: {FEATURES_PATH}')
print(f'Категориальные признаки сохранены: {CATEGORICAL_FEATURES_PATH}')
print(f'Важность признаков сохранена: {IMPORTANCE_PATH}')
display(feature_importance.head(20))

del train_pool, early_stop_pool
gc.collect()

for path in (TRAIN_FEATURES_PATH, EARLY_STOP_FEATURES_PATH):
    path.unlink()
print('Временные файлы признаков удалены.')


Строк для обучения: 7,645,843
Строк для ранней остановки: 851,647
0:	test: 0.8908044	best: 0.8908044 (0)	total: 6.38s	remaining: 1h 46m 15s
100:	test: 0.9375224	best: 0.9375224 (100)	total: 9m 2s	remaining: 1h 20m 29s
200:	test: 0.9398006	best: 0.9398006 (200)	total: 18m 17s	remaining: 1h 12m 42s
300:	test: 0.9403485	best: 0.9403485 (300)	total: 27m 37s	remaining: 1h 4m 8s
400:	test: 0.9406652	best: 0.9406652 (400)	total: 36m 34s	remaining: 54m 37s
500:	test: 0.9408519	best: 0.9408519 (500)	total: 45m 43s	remaining: 45m 33s
600:	test: 0.9410524	best: 0.9410525 (598)	total: 54m 47s	remaining: 36m 22s
700:	test: 0.9411604	best: 0.9411604 (700)	total: 1h 4m 6s	remaining: 27m 20s
800:	test: 0.9412390	best: 0.9412390 (800)	total: 1h 13m 11s	remaining: 18m 10s
900:	test: 0.9413180	best: 0.9413180 (900)	total: 1h 22m 25s	remaining: 9m 3s
999:	test: 0.9413620	best: 0.9413620 (999)	total: 1h 31m 37s	remaining: 0us

bestTest = 0.9413620265
bestIteration = 999

Лучшая итерация: 999
Метрики обучен

,feature,importance
30,event_type,21.309836
28,i2v_session_sim,12.849023
0,candidate_rank,11.396304
1,candidate_rank_norm,9.345087
29,i2v_last_sim,7.449324
33,repeat_ratio,4.194064
32,session_unique,3.384412
27,distance_from_end,2.825121
9,type_count,2.276445
16,item_cart_after_last_order,2.199275


Временные файлы признаков удалены.


## Сравнение этапов на train и validation

Для диагностического score на train используется до 500 000 сессий только из части `train`, выделенной для обучения. Сессии `early_stop` в эту оценку не входят. Доли групп будущих действий, длины истории и дней сохраняются с округлением квот.

Для validation используется до 1 000 000 сессий, в которых начиная с временной границы есть хотя бы один cart и хотя бы один order. Действия до границы не участвуют в этом отборе; оба действия могут относиться к разным товарам. Наличие истории до границы не обязательно. Если подходящих сессий меньше лимита, берутся все; иначе выбирается ровно указанный максимум с распределением по всему порядку файла. Эта оценка относится к сессиям с carts и orders в последней неделе и не описывает качество на всех типах сессий.

Сессии читаются из `evaluation_sessions.jsonl` по одной. Для каждого типа действий загружается одна Item2Vec-модель, а CatBoost оценивает кандидатов небольшими порциями. Top-20 выбирается по массиву предсказаний. Для validation-сессий с пустой историей рекомендации рассчитываются один раз на тип действия; попадания считаются отдельно по целям каждой сессии.

Популярный baseline, кандидатогенерация и CatBoost сравниваются на одинаковых сессиях. Для clicks, carts и orders считаются Recall@20 популярного baseline и CatBoost, покрытие целей среди 150 кандидатов, количество попаданий и сессий с целью. Итоговый score — сумма Recall@20 с весами 0.10, 0.30 и 0.60. В таблице указывается фактический размер каждой выборки. Сессии без будущих clicks не участвуют в метрике clicks.

В этой версии train по-прежнему делится на историю и цели в пропорции 80/20, а validation — по общей временной границе. Новая схема меняет отбор сессий для обучения. После изменения параметров нужно повторно выполнить подсчёт размеров, подготовку данных и обучение, чтобы новые выборки и модель соответствовали друг другу.


In [7]:
STAGE_ORDER = [
    'Popular baseline@20',
    'Candidate generation@150',
    'CatBoost rerank@20',
]
SPLIT_ORDER = ['train_sample', 'validation']

metric_counts = {
    split_name: {
        stage: {
            event_type: {
                'score_sum': 0.0,
                'sessions': 0,
                'hits': 0,
                'denominator': 0,
                'target_items': 0,
            }
            for event_type in EVENT_TYPES
        }
        for stage in STAGE_ORDER
    }
    for split_name in SPLIT_ORDER
}


def update_metric(split_name, stage, event_type, predictions, targets):
    if not targets:
        return

    if stage == 'Candidate generation@150':
        denominator = len(targets)
        top_k = CANDIDATE_K
    else:
        denominator = min(TOP_K, len(targets))
        top_k = TOP_K

    hits = len(set(predictions[:top_k]) & targets)
    values = metric_counts[split_name][stage][event_type]
    values['score_sum'] += hits / denominator
    values['sessions'] += 1
    values['hits'] += hits
    values['denominator'] += denominator
    values['target_items'] += len(targets)


def flush_eval_batch(eval_rows, eval_groups, model):
    if not eval_rows:
        return

    frame = downcast_frame(pd.DataFrame.from_records(eval_rows))
    scores = model.predict_proba(frame[feature_cols], thread_count=4)[:, 1]

    for split_name, event_type, targets, candidates, start, stop in eval_groups:
        top_indices = np.argsort(-scores[start:stop], kind='stable')[:TOP_K]
        ranked_aids = [candidates[int(position)] for position in top_indices]
        update_metric(
            split_name, 'CatBoost rerank@20', event_type, ranked_aids, targets
        )

    eval_rows.clear()
    eval_groups.clear()


evaluated_sessions = dict(evaluation_session_counts)
empty_history_predictions = {}
completed_event_types = []

for event_type in EVENT_TYPES:
    print(f'Оценка {event_type}: загрузка Item2Vec и построение индекса.')
    model, index, idx_to_aid = load_model_and_build_index(
        MODELS_DIR / f'item2vec_{event_type}.model'
    )
    eval_rows = []
    eval_groups = []
    popular_top = popular_items[event_type][:TOP_K]
    scored_sessions = 0

    def queue_eval_session(split_name, history_events, targets, prediction_ts):
        cached = (
            empty_history_predictions.get(event_type)
            if not history_events else None
        )
        if cached is not None:
            candidates, ranked_aids = cached
            update_metric(
                split_name, 'Popular baseline@20', event_type, popular_top, targets
            )
            update_metric(
                split_name, 'Candidate generation@150', event_type, candidates, targets
            )
            update_metric(
                split_name, 'CatBoost rerank@20', event_type, ranked_aids, targets
            )
            return

        history_aids = [str(event['aid']) for event in history_events]
        history_types = [event['type'] for event in history_events]
        history_ts = [float(event['ts']) for event in history_events]

        candidates, candidate_metadata = generate_candidates(
            history_aids, history_types, history_ts, event_type,
            model, index, idx_to_aid,
            popular_items=popular_items, type_weights=TYPE_WEIGHTS,
            top_k=CANDIDATE_K, last_n=LAST_N,
            return_metadata=True,
        )
        update_metric(
            split_name, 'Popular baseline@20', event_type, popular_top, targets
        )
        update_metric(
            split_name, 'Candidate generation@150', event_type, candidates, targets
        )

        feature_rows = compute_features_for_session(
            history_aids, history_types, history_ts,
            candidates, event_type, model,
            prediction_ts=prediction_ts,
            candidate_metadata=candidate_metadata,
        )
        if not history_events:
            frame = downcast_frame(pd.DataFrame.from_records(feature_rows))
            scores = catboost_model.predict_proba(
                frame[feature_cols], thread_count=4
            )[:, 1]
            top_indices = np.argsort(-scores, kind='stable')[:TOP_K]
            ranked_aids = [candidates[int(position)] for position in top_indices]
            empty_history_predictions[event_type] = (candidates, ranked_aids)
            update_metric(
                split_name, 'CatBoost rerank@20', event_type, ranked_aids, targets
            )
            return

        start = len(eval_rows)
        eval_rows.extend(feature_rows)
        eval_groups.append((
            split_name, event_type, targets, candidates, start, len(eval_rows)
        ))

    with open(EVAL_SESSIONS_PATH, 'r') as eval_file:
        for line in eval_file:
            record = ujson.loads(line)
            targets = set(record['targets'][event_type])
            if not targets:
                continue

            queue_eval_session(
                record['split'], record['history'], targets,
                prediction_ts=record['prediction_ts'],
            )
            scored_sessions += 1

            if len(eval_groups) >= EVAL_BATCH_SESSIONS:
                flush_eval_batch(eval_rows, eval_groups, catboost_model)

            if scored_sessions % 5_000 == 0:
                print(f'{event_type}: оценено {scored_sessions:,} сессий с целью.')

    flush_eval_batch(eval_rows, eval_groups, catboost_model)
    completed_event_types.append(event_type)
    with open(OUT_DIR / 'evaluation_progress.json', 'w') as progress_file:
        json.dump({
            'completed_event_types': completed_event_types,
            'sessions_evaluated': evaluated_sessions,
            'metric_counts': metric_counts,
        }, progress_file, indent=2)
    print(f'Оценка завершена: {event_type}; сессий с целью — {scored_sessions:,}.')
    del model, index, idx_to_aid
    gc.collect()

result_rows = []
for split_name in SPLIT_ORDER:
    for stage in STAGE_ORDER:
        is_coverage = stage == 'Candidate generation@150'
        row = {
            'split': split_name,
            'stage': stage,
            'metric': (
                'candidate coverage@150' if is_coverage else 'Recall@20'
            ),
            'sessions_evaluated': evaluated_sessions[split_name],
        }
        group_scores = {}

        for event_type in EVENT_TYPES:
            values = metric_counts[split_name][stage][event_type]
            score = (
                values['score_sum'] / values['sessions']
                if values['sessions'] else 0.0
            )
            group_scores[event_type] = score
            score_name = (
                f'{event_type}_coverage_at_150'
                if is_coverage else f'{event_type}_recall_at_20'
            )
            row[score_name] = score
            row[f'{event_type}_sessions_with_target'] = values['sessions']
            row[f'{event_type}_hits'] = values['hits']
            row[f'{event_type}_denominator_sum'] = values['denominator']
            row[f'{event_type}_target_items'] = values['target_items']

        weighted_score = sum(
            EVENT_WEIGHTS[event_type] * group_scores[event_type]
            for event_type in EVENT_TYPES
        )
        if is_coverage:
            row['weighted_recall_at_20'] = None
            row['weighted_candidate_coverage_at_150'] = weighted_score
        else:
            row['weighted_recall_at_20'] = weighted_score
            row['weighted_candidate_coverage_at_150'] = None
        result_rows.append(row)

stage_comparison = pd.DataFrame(result_rows)
for row in result_rows:
    print(f"\n{row['split']} — {row['stage']}: {row['metric']}")
    for event_type in EVENT_TYPES:
        score_name = (
            f'{event_type}_coverage_at_150'
            if row['metric'] == 'candidate coverage@150'
            else f'{event_type}_recall_at_20'
        )
        print(
            f"  {event_type}: {row[score_name]:.5f}; "
            f"попадания {row[f'{event_type}_hits']:,}; "
            f"целевых товаров {row[f'{event_type}_target_items']:,}; "
            f"сессий с целью "
            f"{row[f'{event_type}_sessions_with_target']:,}"
        )
    if row['metric'] == 'Recall@20':
        print(f"  Взвешенный Recall@20: {row['weighted_recall_at_20']:.5f}")
    else:
        print(
            '  Взвешенное покрытие кандидатов: '
            f"{row['weighted_candidate_coverage_at_150']:.5f}"
        )

STAGE_REPORT_PATH = OUT_DIR / 'stage_comparison.csv'
stage_comparison.to_csv(STAGE_REPORT_PATH, index=False)
print(f'Сравнение сохранено: {STAGE_REPORT_PATH}')

Оценка clicks: загрузка Item2Vec и построение индекса.
clicks: оценено 5,000 сессий с целью.
clicks: оценено 10,000 сессий с целью.
clicks: оценено 15,000 сессий с целью.
clicks: оценено 20,000 сессий с целью.
clicks: оценено 25,000 сессий с целью.
clicks: оценено 30,000 сессий с целью.
clicks: оценено 35,000 сессий с целью.
clicks: оценено 40,000 сессий с целью.
clicks: оценено 45,000 сессий с целью.
clicks: оценено 50,000 сессий с целью.
clicks: оценено 55,000 сессий с целью.
clicks: оценено 60,000 сессий с целью.
clicks: оценено 65,000 сессий с целью.
clicks: оценено 70,000 сессий с целью.
clicks: оценено 75,000 сессий с целью.
clicks: оценено 80,000 сессий с целью.
clicks: оценено 85,000 сессий с целью.
clicks: оценено 90,000 сессий с целью.
clicks: оценено 95,000 сессий с целью.
clicks: оценено 100,000 сессий с целью.
clicks: оценено 105,000 сессий с целью.
clicks: оценено 110,000 сессий с целью.
clicks: оценено 115,000 сессий с целью.
clicks: оценено 120,000 сессий с целью.
click

## Сохранённые файлы

В папке `/kaggle/working` находятся:

- `catboost_ranker.cbm` — обученная модель;
- `feature_cols.json` — порядок входных признаков;
- `categorical_features.json` — имена категориальных признаков;
- `popular_items.json` — популярные товары по типам событий;
- `catboost_feature_importance.csv` — важность признаков;
- `stage_comparison.csv` — сравнение этапов на train и validation;
- `evaluation_sessions.jsonl` — сессии для расчёта итоговых метрик;
- `evaluation_progress.json` — накопленные результаты после каждого типа действий.
- `train_selected_sessions.jsonl` — выбранные сессии, их группы и назначение;
- `training_sample_summary.csv` — квоты и фактическое распределение сессий;
- `training_candidates_summary.csv` — цели, найденные позитивы и число строк для обучения и ранней остановки.
